# **Model Definition**

Import basic **Libraries**

In [1]:
from tensorflow import keras
from tensorflow.keras import layers, Sequential
import numpy as np
import matplotlib.pyplot as plt

Import Dataset

In [2]:
MNIST = keras.datasets.mnist

1. **Normalising** to 0-1 by dividing by 255
2. y_test and y_Train not used anywhere further, because for autoencoders it is not required.

In [3]:
(x_train, y_train), (x_test, y_test) = MNIST.load_data()
x_train = x_train / 255.0
x_test = x_test / 255.0

In [4]:
x_train = x_train.reshape(-1, 784)
x_test  = x_test.reshape(-1, 784)

Define the autoencoder

In [5]:
model = Sequential()

Encoder

In [6]:
model.add(layers.Dense(25, activation='relu'))
model.add(layers.BatchNormalization())

model.add(layers.Dense(25, activation='relu'))
model.add(layers.BatchNormalization())

model.add(layers.Dense(25, activation='relu'))
model.add(layers.BatchNormalization())


Bottleneck

In [7]:
model.add(layers.Dense(10, activation='relu'))

Decoder

In [8]:
model.add(layers.Dense(25, activation='relu'))
model.add(layers.BatchNormalization())

model.add(layers.Dense(25, activation='relu'))
model.add(layers.BatchNormalization())

model.add(layers.Dense(784, activation='sigmoid'))

In [9]:
model.compile(
    optimizer='adam',
    loss='mse'
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ ?                      │   0 (unbuilt) │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ ?                      │   0 (unbuilt) │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ ?                      │   0 (unbuilt) │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ ?                      │   0 (unbuilt) │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ ?                      │   0 (unbuilt) │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [10]:
model.fit(x_train, x_train)

1875/1875 ━━━━━━━━━━━━━━━━━━━━ 13s 4ms/step - loss: 0.0920


# **Anomaly detection**

Using functions for compact use in future
plotRecon plot original and reconstructed

In [11]:
import matplotlib.pyplot as plt
def plotRecon(img, img_recon):
  plt.subplot(1,2,1)
  plt.title("Original")
  plt.imshow(img.reshape(28,28), cmap="gray")
  plt.axis("off")

  plt.subplot(1,2,2)
  plt.title("Reconstruction")
  plt.imshow(img_recon.reshape(28,28), cmap="gray")
  plt.axis("off")

  plt.show()

1. process the data to match the required model prerequiste
2. Note that : 255 - img --> used for inverting the image to match mnist set (black on white)

In [12]:
def imgProcess(img):
    img = cv2.resize(img, (28, 28))
    img = 255 - img
    img = img / 255.0
    img = img.reshape(1, 784)

    img_recon = model.predict(img)

    error = np.mean((img - img_recon)**2)

    return error,plotRecon(img,img_recon)

Mnist  reconstruction error for a **benchmark**.

In [13]:
x_test_recon = model.predict(x_test)
mnist_error = np.mean((x_test - x_test_recon)**2, axis=1)

print("Mean MNIST reconstruction error:", mnist_error.mean())


313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
Mean MNIST reconstruction error: 0.03348675282196826


Imports malayalam fonts and calls imgProcess function for each

In [14]:
import cv2
import numpy as np

img1 = cv2.imread("/content/malayalam-ref001.png", cv2.IMREAD_GRAYSCALE)
print(imgProcess(img1))

img2 = cv2.imread("/content/malayalam-ref002.png", cv2.IMREAD_GRAYSCALE)
print(imgProcess(img2))

img3 = cv2.imread("/content/malayalam-ref003.png", cv2.IMREAD_GRAYSCALE)
print(imgProcess(img3))

img4 = cv2.imread("/content/malayalam-ref004.png", cv2.IMREAD_GRAYSCALE)
print(imgProcess(img4))

img5 = cv2.imread("/content/malayalam-ref005.png", cv2.IMREAD_GRAYSCALE)
print(imgProcess(img5))

error: OpenCV(4.13.0) /io/opencv/modules/imgproc/src/resize.cpp:4208: error: (-215:Assertion failed) !ssize.empty() in function 'resize'
